# How to Compute the Conjunctive Normal Form

We use the class `RecursiveSet`.  This class implements sets in a way such that sets can be nested and the elements of sets are compared structurally and not by reference.

In [ ]:
%load ../lib/RecursiveSet.jsh

Formulas are represented as trees of objects.  The record classes used to represent formulas are defined in the file `Formula.jsh`.  In order to convert a string into a formula we use the *parser* that is defined in the file `PropositionalLogicParser.jsh`.

In [ ]:
%load Formula.jsh
%load PropositionalLogicParser.jsh

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

## Literals and Clauses

A *literal* is either a propositional variable or the negation of a propositional variable.  In the file `Formula.jsh`, literals are represented as follows:
```java
interface Literal {}

record Var(String name) implements Formula, Literal { ... }
record Neg(String name) implements Literal { ... }
```
Hence a propositional variable `p` is represented as `new Var("p")`.  As every propositional variable is a literal, the record `Var` implements both the interface `Formula` and the interface `Literal`.  The negated variable $\neg p$ is represented as `new Neg("p")` when it is used as a literal.

A *clause* is a set of literals and is therefore represented as a `RecursiveSet<Literal>`.  A formula in *conjunctive normal form* is a set of clauses and is therefore represented as a `RecursiveSet<RecursiveSet<Literal>>`.  Unfortunately, Java does not support *type aliases*.  Therefore, we have to write these types in full.

In [ ]:
RecursiveSet<Literal> C = set(new Var("p"), new Neg("q"));
C

In [ ]:
var f = parse("p ↔ q");
f

The function `purify(f)` takes a formula `f` from propositional logic and eliminates all occurrences of the operator `'↔'` and the operator `'→'` from this formula.  This is done by using the following equivalences:
$$ \begin{array}{lrcl}
   (a) & g \leftrightarrow h &\Leftrightarrow& (\neg g \vee h) \wedge (\neg h \vee g) \\
   (b) & g \rightarrow h &\Leftrightarrow& \neg g \vee h 
   \end{array}
$$
The resulting formula is called a *pure* or *arrow free* formula.

In TypeScript we were able to define a type `Pure` that describes *pure* formulas.  In Java, all formulas have the type `Formula`.  Hence the fact that the result of `purify` does not contain the operators `↔` and `→` is not checked by the compiler.

In the code below, the case `Var _, Verum _, Falsum _` matches if `f` is a variable or one of the constants $\top$ and $\bot$.  The character `_` is an *unnamed variable*.  In this case, the formula `f` is returned unchanged.

In [ ]:
Formula purify(Formula f) {
    return switch (f) {
        case Var _, Verum _, Falsum _ -> f;
        case Not(var g)               -> new Not(purify(g));
        case And(var g, var h)        -> new And(purify(g), purify(h));
        case Or(var g, var h)         -> new Or(purify(g), purify(h));
        case Implies(var g, var h)    -> new Or(new Not(purify(g)), purify(h));
        case Equiv(var g, var h)      -> {
            var gs = purify(g);
            var hs = purify(h);
            yield new And(new Or(new Not(gs), hs), new Or(new Not(hs), gs));
        }
        default -> throw new IllegalArgumentException("unknown formula: " + f);
    };
}

In [ ]:
var f1 = purify(f);
f1

The function $\texttt{nnf}(f)$ computes the *negation normal form* of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$.  The expression $\texttt{nnf}(f)$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
</ol>
The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
     <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
</ol>

The functions `nnf` and `neg` are mutually recursive.  In JShell, a method may refer to a method that is defined later.  Nevertheless, we define both functions in a single cell.

In [ ]:
Formula nnf(Formula f) {
    return switch (f) {
        case Var _, Verum _, Falsum _ -> f;
        case Not(var g)               -> neg(g);
        case And(var g, var h)        -> new And(nnf(g), nnf(h));
        case Or(var g, var h)         -> new Or(nnf(g), nnf(h));
        default -> throw new IllegalArgumentException("formula is not pure: " + f);
    };
}

Formula neg(Formula f) {
    return switch (f) {
        case Var v             -> new Not(v);
        case Verum _           -> new Falsum();
        case Falsum _          -> new Verum();
        case Not(var g)        -> nnf(g);
        case And(var g, var h) -> new Or(neg(g), neg(h));
        case Or(var g, var h)  -> new And(neg(g), neg(h));
        default -> throw new IllegalArgumentException("formula is not pure: " + f);
    };
}

In [ ]:
var f2 = nnf(f1);
f2

The function $\texttt{cnf}(f)$ takes a formula $f$ that is in *negation normal form*, i.e. the negation operator is only applied to propositional variables and returns the *conjunctive normal form* of $f$ in *set notation*.  In order to achieve
this it uses the distributive law
$$ (f \wedge g) \vee (h \wedge k) \Leftrightarrow (f \vee h) \wedge (f \vee k) \wedge (g \vee h) \wedge (g \vee k). $$

The case `Not(Var(var p))` uses a *nested record pattern*: It matches if `f` is the negation of a variable.  In this case, `p` is bound to the name of this variable.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> cnf(Formula f) {
    return switch (f) {
        case Var v             -> set(set(v));
        case Verum _           -> set();
        case Falsum _          -> set(set());
        case Not(Var(var p))   -> set(set(new Neg(p)));
        case And(var g, var h) -> cnf(g).union(cnf(h));
        case Or(var g, var h)  -> {
            var gs = cnf(g);
            var hs = cnf(h);
            yield flatMap(gs, c -> hs.map(d -> c.union(d)));
        }
        default -> throw new IllegalArgumentException("formula is not in NNF: " + f);
    };
}

In [ ]:
cnf(f2)

The function `getComplement` computes the complement $\overline{l}$ of a literal $l$. 
If $p$ is a propositional variable, the complement is defined as follows:
 * $\overline{p} = \neg p$,
 * $\overline{\neg p} = p$.  

In [ ]:
Literal getComplement(Literal l) {
    return switch (l) {
        case Var(var p) -> new Neg(p);
        case Neg(var p) -> new Var(p);
        default -> throw new IllegalArgumentException("unknown literal: " + l);
    };
}

The function $\texttt{isTrivial}(C)$ checks whether the clause $C$ is *trivial*.

In [ ]:
boolean isTrivial(RecursiveSet<Literal> clause) {
    return clause.some(l -> clause.contains(getComplement(l)));
}

The function $\texttt{simplify}(Cs)$ takes a set of clauses and removes all trivial clauses from $Cs$.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> simplify(RecursiveSet<RecursiveSet<Literal>> clauses) {
    return clauses.filter(c -> !isTrivial(c));
}

We know that the following equivalence holds:
$$ C \wedge (C \vee D) \Leftrightarrow C $$ 
For this reason, if we have a set $S$ of clauses containing to clauses $C_1$ and $C_2$ such that
$$ C_1 \subset C_2,$$
then we can drop the clause $C_2$ from $S$ since it is *subsumed* by $C_1$.

The function `findSubsumed` takes a set of clauses $S$ and a clause $C_1$ and returns the set of all clauses $C_2$ from $S$ that are subsumed by $C_1$.  These clauses will later be removed from $S$.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> findSubsumed(RecursiveSet<RecursiveSet<Literal>> S,
                                                 RecursiveSet<Literal> C1) {
    return S.filter(C2 -> !C2.equals(C1) && C1.isSubset(C2));
}

Given a set $S$ of clauses, the function `subsume` removes those clauses from $S$ that are subsumed by another clause in $S$.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> subsume(RecursiveSet<RecursiveSet<Literal>> S) {
    return S.difference(flatMap(S, cl -> findSubsumed(S, cl)));
}

The function $\texttt{normalize}$ takes a propositional formula $f$ and transforms $f$ into *conjunctive normal form*.  
Furthermore, both *trivial* clauses and clauses that are *subsumed* by simpler clauses are removed.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> normalize(Formula f) {
    var pure    = purify(f);           // Formula -> pure formula
    var negNF   = nnf(pure);           // pure formula -> negation normal form
    var clauses = cnf(negNF);          // negation normal form -> set of clauses
    return subsume(simplify(clauses)); // remove trivial and subsumed clauses
}

As literals and sets are printed in a readable way, there is no need for a function that converts a set of clauses into a string.  The function `test` takes a string, parses it as a formula, and prints the conjunctive normal form of this formula.

In [ ]:
void test(String s) {
    var f = parse(s);
    System.out.println("The CNF of " + s + " is:");
    System.out.println(normalize(f));
}

In [ ]:
test("(¬p → q) → (p → q) → q");

In [ ]:
test("(a → b) ↔ (¬a ∧ ¬b)");

In [ ]:
test("(p ∧ q → r) ∨ ¬r → ¬p");

In [ ]:
test("(p ∧ q → r) ∨ ¬r → ¬p ↔ ¬p");

In [ ]:
test("⊤");

In [ ]:
test("⊥");

In [ ]:
test("p → q");

In [ ]:
test("p ∧ (p → q)");

In [ ]:
test("(p ∧ q) → r");

In [ ]:
test("p ↔ q");

In [ ]:
test("(p → q) ∧ (q → r)");

In [ ]:
test("¬(p ∨ q)");

In [ ]:
test("p ∧ p");

The functions defined in this notebook are also available in the file `04-CNF.jsh`.  This file is used by later notebooks to transform formulas into conjunctive normal form.